In [2]:
from understatapi import UnderstatClient
import pandas as pd
import requests

from thefuzz import process, fuzz

In [8]:
players_raw = pd.read_csv('https://raw.githubusercontent.com/ilyandho/FPL-Optimal-Transfer/refs/heads/FantasyGo/FPL%20predictors/with%20new%20features/data/vaastav/data/2025-26/players_raw.csv')
players_1 = pd.read_csv('https://raw.githubusercontent.com/ilyandho/FPL-Optimal-Transfer/refs/heads/FantasyGo/FPL%20predictors/with%20new%20features/data/vaastav/data/2025-26/gws/gw1.csv')
v = pd.read_csv('https://github.com/ilyandho/FPL-Optimal-Transfer/raw/refs/heads/FantasyGo/FPL%20predictors/with%20new%20features/data/vaastav/data/2025-26/players/Aaron_Hickey_116/gw.csv')
master_fpl_understat_map = pd.read_csv('https://raw.githubusercontent.com/ChrisMusson/FPL-ID-Map/main/Master.csv')

# player_gw_stats = pd.read_csv('https://github.com/olbauday/FPL-Core-Insights/raw/refs/heads/main/data/2025-2026/By%20Tournament/Premier%20League/GW1/player_gameweek_stats.csv')
# playerstats_1 = pd.read_csv('https://github.com/olbauday/FPL-Core-Insights/raw/refs/heads/main/data/2025-2026/By%20Tournament/Premier%20League/GW1/playerstats.csv')
players = pd.read_csv('https://github.com/olbauday/FPL-Core-Insights/raw/refs/heads/main/data/2025-2026/By%20Tournament/Premier%20League/GW1/players.csv')
teams = pd.read_csv('https://github.com/olbauday/FPL-Core-Insights/raw/refs/heads/main/data/2025-2026/By%20Tournament/Premier%20League/GW1/teams.csv')

git_data_olbauday_base = "https://github.com/olbauday/FPL-Core-Insights/raw/refs/heads/main/data/2025-2026/By%20Tournament/Premier%20League/"

fpl_fixtures_url = 'https://fantasy.premierleague.com/api/fixtures/?event='

player_summary = 'https://fantasy.premierleague.com/api/element-summary/'

# bootstrap = requests.get('https://fantasy.premierleague.com/api/bootstrap-static/').json()
# bootstrap

## Understat


In [13]:
# Initialize the client
with UnderstatClient() as understat:
    # Use .league() to specify the league, then .get_player_data() for the season
    data = understat.league(league="EPL").get_player_data(season="2025")

# This will return a list of dictionaries containing player stats
understat_data = pd.DataFrame(data)
understat_data.to_csv('./data/understat/understat_data_19.csv')
understat_data

,id,player_name,games,time,goals,xG,assists,xA,shots,key_passes,yellow_cards,red_cards,position,team_title,npg,npxG,xGChain,xGBuildup
0,8260,Erling Haaland,20,1756,19,18.415004886686802,4,3.0899876076728106,75,11,0,0,F,Manchester City,18,16.89266712218523,20.211640633642673,2.778261484578252
1,13222,Thiago,20,1680,14,13.802449688315392,1,1.5831863638013601,44,9,3,0,F S,Brentford,9,9.235436581075191,11.383689273148775,2.6555524803698063
2,11363,Antoine Semenyo,19,1710,9,7.810670031234622,3,2.4660277236253023,47,25,5,0,M,Bournemouth,8,6.288332333788276,10.328016273677349,2.9676714949309826
3,501,Danny Welbeck,19,1107,8,6.669347804039717,0,0.31755480915308,28,10,3,0,F S,Brighton,7,4.385841159150004,6.039988946169615,1.8120669340714812
4,5555,Dominic Calvert-Lewin,18,1224,8,7.4578270222991705,0,1.2653321214020252,36,12,0,0,F S,Leeds,7,6.696658169850707,8.989225076511502,1.3431714698672295
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
480,14198,Shea Lacey,1,2,0,0.01579156517982483,0,0,1,0,0,0,S,Manchester United,0,0.01579156517982483,0.01579156517982483,0
481,14219,Mohamadou Kanté,3,8,0,0.06551869213581085,0,0.10989412665367126,1,1,0,0,S,West Ham,0,0.06551869213581085,0.17541281878948212,0
482,14263,Joél Drakes-Thomas,2,2,0,0,0,0,0,0,0,0,S,Crystal Palace,0,0,0.3753929138183594,0.3753929138183594
483,14266,Bendito Mantato,1,14,0,0,0,0.03647809848189354,0,1,0,0,S,Manchester United,0,0,0.12379638105630875,0.12379638105630875


In [14]:

# List of IDs from your data
player_ids = understat_data['id'].values #['8260', '13222', '11363'] # Haaland, Thiago, Semenyo, etc.
player_name_id = understat_data.set_index('id')['player_name']
player_name_id
all_history = []

with UnderstatClient() as understat:
    for p_id in player_ids:
        # This gets the season-by-season history for that specific ID
        history = understat.player(player=p_id).get_match_data()

        # # Add the player name back in so you know who is who
        for gw in history:
            gw['understat_id'] = p_id
            gw['understat_name'] = player_name_id[p_id]
            all_history.append(gw)

# # Convert to a history DataFrame
history_df = pd.DataFrame(all_history)
history_df.to_csv('./data/understat/understat_hist_19.csv', index=False)

## FPL


### GW 1


In [9]:
all_player_gw_stats = []
for gw in range(1,21):
    player_gw_stats = pd.read_csv(f'{git_data_olbauday_base}GW{gw}/player_gameweek_stats.csv')
    player_gw_stats['round'] = gw
    all_player_gw_stats.append(player_gw_stats)

# 2. Combine all DataFrames at once (much faster than looping concat)
all_player_gw_stats = pd.concat(all_player_gw_stats, ignore_index=True)

all_teams = []
for gw in range(1,21):
    team_stats = pd.read_csv(f'{git_data_olbauday_base}GW{gw}/teams.csv')
    team_stats['round'] = gw
    all_teams.append(team_stats)

all_team_stats = pd.concat(all_teams, ignore_index=True)

all_players = []
for gw in range(1,21):
    team_stats = pd.read_csv(f'{git_data_olbauday_base}GW{gw}/players.csv')
    team_stats['round'] = gw
    all_players.append(team_stats)

players = pd.concat(all_players, ignore_index=True)

matches = []

for gw in range(1,21):
    data = requests.get(fpl_fixtures_url+str(gw)).json()

    matches = [*matches, *[{
                    'round': event['event'], 'team_id': event['id'], 'team_a': event['team_a'], 'team_h': event['team_h'],
                    'team_h_difficulty':event['team_h_difficulty'], 'team_a_difficulty':event['team_a_difficulty'], 'team_a_score': event['team_a_score'],
                    'team_h_score': event['team_h_score'], 'kickoff_time': event['kickoff_time']
                     } for event in data]]
    #     []
    # matches.append()
match_details = pd.DataFrame(matches)
match_details.to_csv('./data/match_details_19.csv', index=False)

In [10]:

player_gw_stats_clean = all_player_gw_stats[all_player_gw_stats['status']!= 'u']  # Remove unavailable players
player_gw_stats_clean.columns.tolist()
gw_stats_cols =  [
                    'id', 'first_name', 'second_name', 'web_name', 'now_cost',  'selected_by_percent',  'form',  'event_points', 'transfers_in_event',
                    'transfers_out_event', 'value_form', 'ep_next', 'ep_this', 'chance_of_playing_next_round', 'chance_of_playing_this_round',  'gw',
                    'total_points', 'minutes',  'goals_scored',  'assists',  'clean_sheets',  'goals_conceded', 'yellow_cards',  'red_cards',  'saves',
                    'starts',  'bonus',  'bps',  'transfers_in',  'transfers_out', 'expected_goals',  'expected_assists',  'expected_goal_involvements',
                    'expected_goals_conceded',  'influence',  'creativity',  'threat',  'ict_index',  'tackles',  'clearances_blocks_interceptions',
                    'recoveries',  'defensive_contribution',  'round',

                    # 'expected_goals_per_90', 'expected_assists_per_90', 'expected_goal_involvements_per_90',
                    # 'expected_goals_conceded_per_90', 'saves_per_90', 'clean_sheets_per_90', 'goals_conceded_per_90', 'defensive_contribution_per_90',

                    # 'status', 'news', 'news_added', 'now_cost_rank', 'now_cost_rank_type', 'selected_rank', 'selected_rank_type', 'form_rank', 'form_rank_type',
                    # 'cost_change_event', 'cost_change_event_fall', 'cost_change_start', 'cost_change_start_fall', 'value_season', 'points_per_game',
                    # 'points_per_game_rank', 'points_per_game_rank_type', 'influence_rank', 'influence_rank_type', 'creativity_rank', 'creativity_rank_type',
                    # 'threat_rank', 'threat_rank_type', 'ict_index_rank', 'ict_index_rank_type', 'corners_and_indirect_freekicks_order', 'direct_freekicks_order',
                    # 'penalties_order', 'set_piece_threat', 'corners_and_indirect_freekicks_text', 'direct_freekicks_text', 'penalties_text',  'own_goals', '
                    # penalties_saved', 'penalties_missed', 'dreamteam_count', 'starts_per_90',
                ]

player_gw_stats_clean = player_gw_stats_clean[gw_stats_cols]
player_gw_stats_clean [['chance_of_playing_this_round', 'chance_of_playing_next_round']]= player_gw_stats_clean[['chance_of_playing_this_round', 'chance_of_playing_next_round']].fillna(100)
ids_with_details = players['player_id'].unique().tolist()
player_gw_stats_clean = player_gw_stats_clean[player_gw_stats_clean['id'].isin(ids_with_details)]
player_gw_stats_clean = player_gw_stats_clean.merge(players[['player_id','team_code', 'position', 'round']], left_on=['id', 'round'], right_on=['player_id', 'round'], how='left')
player_gw_stats_clean = player_gw_stats_clean.dropna(subset=['team_code'])
team_data = all_team_stats.rename({'code': 'team_code', 'id': 'team_id', 'name':'team', 'short_name':'team_short_name', 'ep_this': 'xP', 'ep_next':'xP_next'}, axis=1)
player_gw_stats_clean = player_gw_stats_clean.merge(team_data[[
  'team_code', 'team_id', 'team', 'round', 'team_short_name', 'elo', 'strength', 'strength_overall_home', 'strength_overall_away', 'strength_attack_home', 'strength_attack_away',
    'strength_defence_home', 'strength_defence_away']], on=['round','team_code'], how='left')

# Reshape matches so every team_id has its own row per match
match_details = match_details.rename(columns={'team_id': 'match_id'})
matches_melted = match_details.melt(
    id_vars=['match_id', 'round', 'team_h_difficulty', 'team_a_difficulty', 'team_a_score', 'team_h_score', 'kickoff_time'],
    value_vars=['team_a', 'team_h'],
    var_name='side',
    value_name='team_id'
)

player_match_df = player_gw_stats_clean.merge(matches_melted, on=['team_id', 'round'], how='left')
player_match_df

,id,first_name,second_name,web_name,now_cost,selected_by_percent,form,event_points,transfers_in_event,transfers_out_event,...,strength_attack_away,strength_defence_home,strength_defence_away,match_id,team_h_difficulty,team_a_difficulty,team_a_score,team_h_score,kickoff_time,side
0,1,David,Raya Martín,Raya,6.0,36.9,3.2,10,418466,56691,...,1350,1290,1300,9,4,3,1,0,2025-08-17T15:30:00Z,team_a
1,2,Kepa,Arrizabalaga Revuelta,Arrizabalaga,4.1,0.4,0.0,0,793,2256,...,1350,1290,1300,9,4,3,1,0,2025-08-17T15:30:00Z,team_a
2,4,Tommy,Setford,Setford,3.9,0.2,0.0,0,2397,1906,...,1350,1290,1300,9,4,3,1,0,2025-08-17T15:30:00Z,team_a
3,5,Gabriel,dos Santos Magalhães,Gabriel,6.2,14.0,0.0,6,5927,342245,...,1350,1290,1300,9,4,3,1,0,2025-08-17T15:30:00Z,team_a
4,6,William,Saliba,Saliba,6.0,10.3,0.5,9,25445,160545,...,1350,1290,1300,9,4,3,1,0,2025-08-17T15:30:00Z,team_a
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12241,646,João Victor,Gomes da Silva,Gomes,5.3,0.1,2.5,1,175,239,...,1050,1090,1120,200,2,2,0,3,2026-01-03T15:00:00Z,team_h
12242,695,Jackson,Tchatchoua,Tchatchoua,4.4,0.0,1.5,5,89,43,...,1050,1090,1120,200,2,2,0,3,2026-01-03T15:00:00Z,team_h
12243,709,Ladislav,Krejcí,Krejčí,4.5,0.1,3.2,6,818,266,...,1050,1090,1120,200,2,2,0,3,2026-01-03T15:00:00Z,team_h
12244,777,Temple,Ojinnaka,Ojinnaka,4.5,0.0,0.0,0,40,10,...,1050,1090,1120,200,2,2,0,3,2026-01-03T15:00:00Z,team_h


In [ ]:
unique_players = player_gw_stats_clean['id'].unique()

player_api_data = []

for pid in unique_players:
    # Fetch from API
    response = requests.get(f'{player_summary}{pid}/')
    data = response.json()  # assuming JSON response
    history_data = data['history']

    player_api_data = [*player_api_data, *history_data]

pd.DataFrame(player_api_data).rename({'element': 'id'}, axis=1).to_csv('./data/player_summary_19.csv', index=False)


In [13]:
player_api_details

,id,fixture,opponent_team,total_points,was_home,kickoff_time,team_h_score,team_a_score,round,modified,...,starts,expected_goals,expected_assists,expected_goal_involvements,expected_goals_conceded,value,transfers_balance,selected,transfers_in,transfers_out
0,1,9,14,10,False,2025-08-17T15:30:00Z,0.0,1.0,1,False,...,1,0.0,0.00,0.00,1.52,55,0,1531911,0,0
1,1,11,11,6,True,2025-08-23T16:30:00Z,5.0,0.0,2,False,...,1,0.0,0.00,0.00,0.17,55,218659,2284634,277339,58680
2,1,25,12,2,False,2025-08-31T15:30:00Z,1.0,0.0,3,False,...,1,0.0,0.02,0.02,0.52,55,-12311,2406964,146739,159050
3,1,31,16,6,True,2025-09-13T11:30:00Z,3.0,0.0,4,False,...,1,0.0,0.00,0.00,0.20,55,171289,2765759,289041,117752
4,1,41,13,2,True,2025-09-21T15:30:00Z,1.0,1.0,5,False,...,1,0.0,0.01,0.01,0.89,55,-9786,2762632,98100,107886
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13536,656,151,1,0,False,2025-12-13T20:00:00Z,2.0,1.0,16,False,...,0,0.0,0.00,0.00,0.00,49,-78,6501,0,78
13537,656,170,5,0,True,2025-12-20T15:00:00Z,0.0,2.0,17,False,...,0,0.0,0.00,0.00,0.00,49,-55,6447,0,55
13538,656,176,12,0,False,2025-12-27T15:00:00Z,2.0,1.0,18,False,...,0,0.0,0.00,0.00,0.00,49,-29,6413,0,29
13539,656,187,14,0,False,2025-12-30T20:15:00Z,1.0,1.0,19,False,...,0,0.0,0.00,0.00,0.00,49,-31,6388,0,31


In [14]:
player_api_details = pd.read_csv('./data/player_summary_19.csv')
understat_data = pd.read_csv('./data/understat/understat_hist_19.csv')
understat_data = understat_data[understat_data['season'] == 2025]

# ids availabe from the api
live_ids = player_api_details['id'].unique().tolist()
player_match_df_ = player_match_df[player_match_df['id'].isin(live_ids)].copy()

fpl_understat_map = master_fpl_understat_map[(
                                                ~master_fpl_understat_map['25-26'].isna())
                                                & ~(master_fpl_understat_map['understat'].isna())].rename({'25-26': 'fpl_id', 'understat':'understat_id'}, axis=1)

player_match_df_['date'] = pd.to_datetime(player_match_df_['kickoff_time']).dt.date
understat_data['date'] = pd.to_datetime(understat_data['date']).dt.date
player_gw_stats_full = player_match_df_.merge(player_api_details[['id', 'round', 'selected', 'transfers_balance', 'was_home', 'value', 'opponent_team']],
                                            on=['round', 'id'],
                                            how='left')

player_gw_stats_full = player_gw_stats_full.rename(columns={'player_id':'fpl_id'}).drop('id', axis=1)

player_gw_stats_full = player_gw_stats_full[player_gw_stats_full['fpl_id'].isin(fpl_understat_map['fpl_id'].unique().tolist())]

understat_data = understat_data.merge(fpl_understat_map[['fpl_id', 'understat_id']], on='understat_id', how='left').dropna()
player_gw_stats_full = player_gw_stats_full.merge(fpl_understat_map[['fpl_id', 'understat_id']],
                                                  on='fpl_id',
                                                  how='left')

player_gw_stats_full['full_name'] = (player_gw_stats_full['first_name'] + ' ' + player_gw_stats_full['second_name']).str.strip()

player_data = player_gw_stats_full.merge(understat_data[[
                                'goals', 'shots', 'xG','h_team', 'a_team',
                                'h_goals', 'a_goals', 'date', 'season', 'roster_id', 'xA',
                                'key_passes', 'npg', 'npxG', 'xGChain', 'xGBuildup',
                                'understat_id', 'understat_name', 'fpl_id']],
                                on=['date', 'fpl_id', 'understat_id'],
                                how='left'
                                )

player_data = player_data.rename(columns={'ep_this': 'xP', 'ep_next': 'xP_next'})

player_data.to_csv('./data/player_data.csv', index=False)

### Add Odds


In [ ]:
player_data = pd.read_csv('./data/player_data.csv', low_memory=False)
player_data_clean = player_data.dropna()
# Load the current season's data directly from the source
season = "2526" # Change this for historical seasons
url = f"https://www.football-data.co.uk/mmz4281/{season}/E0.csv"

# It's good practice to use a custom User-Agent to avoid blocks
odds_data = pd.read_csv(url, low_memory=False)

# 1. Clean up dates in betting data
odds_data['Date'] = pd.to_datetime(odds_data['Date'], dayfirst=True).dt.date

# 2. Extract key columns (B365H = Bet365 Home Odds, B365D = Draw, B365A = Away)
odds_subset = odds_data[['Date', 'HomeTeam', 'AwayTeam', 'B365H', 'B365D', 'B365A']].copy()
odds_subset = odds_subset.rename(columns={'Date': 'date'})

def add_odds(row):
    win = round(1/row['B365H'], 5)
    draw = round(1/row['B365D'], 5)
    lose = round(1/row['B365A'], 5)

    # Normalize the probabilities (to make the probabilities sum to 100%)
    sum_percent = win + draw + lose
    win_prob = round(win/sum_percent, 3)
    draw_prob = round(draw/sum_percent, 3)
    lose_prob = round(lose/sum_percent, 3)

    return pd.Series([win_prob, draw_prob, lose_prob])

odds_subset[['win_prob', 'draw_prob', 'lose_prob']] = odds_subset.apply(add_odds, axis=1)

# Make sure the names match in the player_data df and odds_subset df
team_map = {
    'Bournemouth': 'Bournemouth',
    'Newcastle': 'Newcastle',
    'Fulham': 'Fulham',
    'West Ham': 'West Ham',
    'Burnley': 'Burnley',
    'Man City': 'Man City',
    'Crystal Palace': 'Crystal Palace',
    'Brentford': 'Brentford',
    'Arsenal': 'Arsenal',
    'Everton': 'Everton',
    'Chelsea': 'Chelsea',
    'Tottenham': 'Spurs',
    'Wolves': 'Wolves',
    'Aston Villa': 'Aston Villa',
    'Sunderland': 'Sunderland',
    'Leeds': 'Leeds',
    "Nott'm Forest": "Nott'm Forest",
    'Brighton': 'Brighton',
    'Man United': 'Man Utd',
    'Liverpool': 'Liverpool'
}

odds_subset['HomeTeam'] = odds_subset['HomeTeam'].map(team_map)
odds_subset['AwayTeam'] = odds_subset['AwayTeam'].map(team_map)

odds_melted = odds_subset.melt(
    id_vars=['date','B365H', 'B365D', 'B365A', 'win_prob', 'draw_prob', 'lose_prob'],
    value_vars=['HomeTeam', 'AwayTeam'],
    var_name='side',
    value_name='team'
)

odds_melted['date'] = odds_melted['date'].astype(str)

player_data_odds = player_data_clean.merge(odds_melted[['date', 'win_prob', 'draw_prob', 'lose_prob','team']],
                  on=['date', 'team'],
                  how='left'
                  )

# 1. Sort by player and round to ensure the sequence is correct
player_data_odds = player_data_odds.sort_values(['fpl_id', 'round'])

# 2. Group by player and apply the difference
player_data_odds['ownership_change'] = player_data_odds.groupby('fpl_id')['selected'].diff().fillna(0)
player_data_odds['pts_bonus'] = player_data_odds['total_points'] - player_data_odds['bonus']

def ownership_change(row):
    net_transfers = row['transfers_in'] - row['transfers_out']
    total_transfers = row['transfers_in'] + row['transfers_out']
    net_transfers_pct = net_transfers / total_transfers if total_transfers != 0 else 0

    return net_transfers_pct

player_data_odds['percenatge_net_transfers'] = player_data_odds.apply(ownership_change, axis=1)

rolling_features = [
    'selected_by_percent', 'form', 'xP','pts_bonus', 'minutes', 'goals_scored', 'assists', 'clean_sheets', 'goals_conceded', 'yellow_cards', 'red_cards',
    'saves', 'starts', 'expected_goals', 'expected_assists', 'expected_goal_involvements', 'expected_goals_conceded', 'influence', 'creativity', 'threat', 'ict_index',
    'tackles', 'clearances_blocks_interceptions', 'recoveries', 'defensive_contribution', 'strength', 'strength_overall_home', 'strength_overall_away',
    'strength_attack_home', 'strength_attack_away', 'strength_defence_home', 'strength_defence_away', 'value', 'goals', 'shots', 'xG','xA', 'key_passes', 'npg', 'npxG',
    'xGChain', 'xGBuildup'
]


# player_data_odds.to_csv('./data/final_player_data.csv')

In [ ]:
sum_features = [
    'creativity', 'influence', 'threat',
    'expected_goals', 'expected_assists', 'expected_goal_involvements',
    'expected_goals_conceded', 'goals_conceded', 'goals_scored',
    'shots', 'key_passes', 'npg', 'npxG',
]

mean_features = [
    'value', 'ict_index', 'selected', 'transfers_in', 'transfers_out'
]

# max_features = [
#     'minutes_1', 'influence'
# ]

identity_features = [
    'whh', 'whd', 'wha',    # bookmaker odds
    'was_home',
    'opponent_team'
]

team_df = play_data.groupby(['team', 'gw']).agg({
    **{f: 'sum' for f in sum_features},
    **{f: 'mean' for f in mean_features},
    # **{f: 'max' for f in max_features},
    **{f: 'first' for f in identity_features}
}).reset_index()

for col in ['expected_goals_conceded_1','goals_conceded_1','goals_scored_1']:
    team_df[f'{col}_3'] = team_df.groupby('team')[col].rolling(3, min_periods=1).mean().reset_index(0, drop=True)
    team_df[f'{col}_5'] = team_df.groupby('team')[col].rolling(5, min_periods=1).mean().reset_index(0, drop=True)

['first_name',
 'second_name',
 'web_name',
 'now_cost',
 'selected_by_percent',
 'form',
 'event_points',
 'transfers_in_event',
 'transfers_out_event',
 'value_form',
 'xP_next',
 'xP',
 'chance_of_playing_next_round',
 'chance_of_playing_this_round',
 'gw',
 'total_points',
 'minutes',
 'goals_scored',
 'assists',
 'clean_sheets',
 'goals_conceded',
 'yellow_cards',
 'red_cards',
 'saves',
 'starts',
 'bonus',
 'bps',
 'transfers_in',
 'transfers_out',
 'expected_goals',
 'expected_assists',
 'expected_goal_involvements',
 'expected_goals_conceded',
 'influence',
 'creativity',
 'threat',
 'ict_index',
 'tackles',
 'clearances_blocks_interceptions',
 'recoveries',
 'defensive_contribution',
 'round',
 'fpl_id',
 'team_code',
 'position',
 'team_id',
 'team',
 'team_short_name',
 'elo',
 'strength',
 'strength_overall_home',
 'strength_overall_away',
 'strength_attack_home',
 'strength_attack_away',
 'strength_defence_home',
 'strength_defence_away',
 'match_id',
 'team_h_difficulty'

In [23]:

cols = [
    'creativity','ict_index', 'influence','selected',  'threat', 'transfers_balance', 'transfers_in', 'transfers_out', 'value', 'was_home',
        'team_h_difficulty', 'team_a_difficulty', 'ownership_change', 'percenatge_net_transfers', 'round', 'fpl_name', 'xP', 'team', 'opponent_team', 'position',

]


player_data_odds[cols]

KeyError: "['fpl_name'] not in index"